In [16]:
pip install langgraph langchain_core langchain_groq tavily-python langchain gradio langchain_community

  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached marshmallow-3.26.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 18.2 MB/s  0:00:00
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached marshmallow-3.26.1-py3-none-any.whl (50 kB)
Using cached typing_inspect-0.9.0-py3-none-any.whl (8.8 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached mypy_extensions-1.1.0-py3-none-any.whl (5.0 kB)

   -- -------------------------------------  1/15 [mypy-extensions]
   ----- -

In [17]:
from typing import Dict, TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain.tools import Tool

from IPython.display import display, Image
from langchain_core.runnables.graph import MermaidDrawMethod
from dotenv import load_dotenv
import os

import sys

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.schema.runnable import RunnableLambda



# Load environment variables and set OpenAI API key
load_dotenv()
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
os.environ["GROQ_API_KEY"] = os.getenv('GROQ_API_KEY')

In [18]:
class State(TypedDict):
    query: str
    response: Optional[str]
    approved: Optional[bool]
    memory: list

In [19]:
model_name = "llama-3.1-8b-instant"

History report generator: System that can search for articles / papers on specific historical topics. Can possibly look into pulling public records for historical information.

In [20]:
tavily_search = TavilySearchResults(k=5)
search_node = RunnableLambda(lambda query: tavily_search.run(query))

def search(state: State) -> State:
    """Search for a website based on the history topic in the query."""

    prompt = ChatPromptTemplate.from_template(
        "Generate a concise search query to find websites about this history topic:\n\n"
        "User query: {query}"
    )

    chain = prompt | ChatGroq(model=model_name, temperature=0)

    results = chain.invoke("chain").content


    search_results = search_node.invoke(refined_query)

    search_results = tavily_client.search(state["query"]) 
    
    state["memory"].append(state["query"]) 
    state["memory"].append(search_results.text) 
    
    return {"results": "search_results"}

C:\Users\ericm\AppData\Local\Temp\ipykernel_37696\779139523.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(k=5)


Agent Architecture: Develop a multi-agent system to plan, research, use tools to gather relevant information, and format the responses in a concise pdf report.

Tool-Calling functionality:

As explored previously, the agent should use tools for performing research and get relevant context from the web, or have a vector database with a large volume of documents to retrieve relevant knowledge.



User Interface and Agent Response:

Final response of the agent should be well structured into a report (pdf format).

A user interface where the user can ask a query and get the response from the LLM, a report that can be downloaded or viewed in the interface.

In [6]:
import streamlit as st
import random
import time

st.write("Streamlit loves LLMs! 🤖 [Build your own chat app](https://docs.streamlit.io/develop/tutorials/llms/build-conversational-apps) in minutes, then make it powerful by adding images, dataframes, or even input widgets to the chat.")

st.caption("Note that this demo app isn't actually connected to any LLMs. Those are expensive ;)")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = [{"role": "assistant", "content": "Let's start chatting! 👇"}]

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Accept user input
if prompt := st.chat_input("What is up?"):
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})
    # Display user message in chat message container
    with st.chat_message("user"):
        st.markdown(prompt)

    # Display assistant response in chat message container
    with st.chat_message("assistant"):
        message_placeholder = st.empty()
        full_response = ""
        assistant_response = random.choice(
            [
                "Hello there! How can I assist you today?",
                "Hi, human! Is there anything I can help you with?",
                "Do you need help?",
            ]
        )
        # Simulate stream of response with milliseconds delay
        for chunk in assistant_response.split():
            full_response += chunk + " "
            time.sleep(0.05)
            # Add a blinking cursor to simulate typing
            message_placeholder.markdown(full_response + "▌")
        message_placeholder.markdown(full_response)
    # Add assistant response to chat history
    st.session_state.messages.append({"role": "assistant", "content": full_response})


2025-10-08 15:32:28.249 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-08 15:32:28.250 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-08 15:32:28.250 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-08 15:32:28.251 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-08 15:32:28.251 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-08 15:32:28.252 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-08 15:32:28.252 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-08 15:32:28.253 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar